# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

This Jupyter notebook guides you through:

- Loading the Croissant metadata and record sets
- Inspecting available record sets and fields (referenced by their `@id`)
- Loading records into pandas DataFrames for analysis
- Performing exploratory data analysis (EDA)
- Visualizing the results

### Dataset Source
The Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load Croissant metadata and inspect base dataset information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset (FAIR² example)
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (not as dict, use attributes)
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Fields in Metadata: {list(dataset.metadata.__dict__.keys())}")

## 2. Data Overview
### Inspecting Available Record Sets
List all available record sets in the dataset and their `@id` values, along with the fields and columns of each.

In [ ]:
# List out all record sets with @id and field info
print("Available record sets in this dataset: (referenced by @id)")
record_sets = dataset.metadata.record_sets if hasattr(dataset.metadata, 'record_sets') else []

for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"    name: {getattr(rs, 'name', None)}")
    # List fields by their @id
    if hasattr(rs, 'fields'):
        field_ids = [f.id for f in rs.fields if hasattr(f, 'id')]
        print(f"    Fields @id: {field_ids}")
    if hasattr(rs, 'columns'):
        col_ids = [col.id for col in rs.columns if hasattr(col, 'id')]
        print(f"    Columns @id: {col_ids}")
    print("")

# Save all record set @id for later extraction
record_set_ids = [rs.id for rs in record_sets]

# If record_sets is empty, print a message with fallback
if not record_sets:
    print("No explicit record sets detected in the schema; inferring from available data files (distribution)...")
    if hasattr(dataset.metadata, 'distributions'):
        for dist in dataset.metadata.distributions:
            print(f"Distribution @id: {dist.id}")

## 2.1. Inspect Sample Record
Show a single record (row) for a selected record set, referencing by `@id`.

**Note:** You can find available record set `@id` and field `@id` values from the previous cell's output.

In [ ]:
# For demonstration, select first available record set @id
if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f"Sample record from record set: {sample_record_set_id}")
    sample_record_iterator = dataset.records(record_set=sample_record_set_id)
    for idx, rec in enumerate(sample_record_iterator):
        print(json.dumps(rec, indent=2))
        if idx > 0:
            break
else:
    print("No record sets found for extracting records.")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for further analysis.

**Reference everything by @id:**

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  Number of records: {len(records)}")
    if records and len(records) > 0:
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")

# If there is at least one DataFrame, show a preview
if dataframes:
    main_record_set_id = record_set_ids[0]
    print(f"\nSample records from main record set @id: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No DataFrames loaded (no record sets detected with records).")

## 4. Exploratory Data Analysis (EDA)
Apply standard EDA techniques. In the FAIR² dataset, typical protein abundance and peptide count fields are **numeric** and useful for filtering and normalization.

We'll filter for proteins with abundance above a threshold (using the appropriate @id), normalize the values, and group by relevant fields when available.

In [ ]:
# Pick record set and field @id from your previous overviews and adjust as needed
import numpy as np

# Example: infer a numeric field name; update this string if your dataset uses different @ids
numeric_field_id = None
group_field_id = None
if dataframes:
    # Choose a numeric field that appears in the main record set
    main_record_set_id = record_set_ids[0]
    cols = dataframes[main_record_set_id].columns
    # Guess a numeric field @id by name substring for demonstration
    for c in cols:
        if "abundance" in c.lower() or "count" in c.lower() or "mw" in c.lower() or "coverage" in c.lower():
            numeric_field_id = c
            break
    # Try group field by 'protein' or 'gene' or 'accession' if present
    for c in cols:
        if "accession" in c.lower() or "gene" in c.lower() or "protein" in c.lower():
            group_field_id = c
            break

    print(f"Selected numeric field for EDA: {numeric_field_id}")
    print(f"Selected group field for EDA:   {group_field_id}")

    df = dataframes[main_record_set_id]
    # Ensure numeric field is numeric type
    if numeric_field_id:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)  # For example, focus on top 25% abundance
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%): {len(filtered_df)} records\n")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optional: group by the group_field_id (if present and not null)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped average {numeric_field_id} by {group_field_id}:\n", grouped.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field, as well as any groupwise summaries, using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id is available, plot group means
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_means.head(10).index, y=group_means.head(10).values, palette="viridis")
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=70)
        plt.show()
else:
    print("Not enough numeric data found for visualization.")

## 6. Conclusion

- This notebook demonstrated standardized exploration of the FAIR² dataset via the Croissant schema, including loading metadata, browsing record sets and fields by `@id`, and extracting and analyzing records.
- All operations referenced specific entities by their Croissant `@id` values for reproducibility and alignment with best practices.
- You can further extend this workflow—e.g., by mapping Croissant fields to domain-specific attributes, or integrating with downstream statistical or machine learning workflows.

_Reference: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json_